In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import torch
import copy
from matplotlib.colors import hsv_to_rgb
import pandas as pd
import matplotlib as mpl
from matplotlib.widgets import LassoSelector
from matplotlib.path import Path
from matplotlib.widgets import Slider
from scipy.ndimage import gaussian_filter1d
import bisect
import sys
sys.path.append(os.path.abspath('..'))
from PIL import Image
import matplotlib.patches as patches

from simu_PSF_polarMFM import *
from extract_experimental_psf import *
from tqdm import tqdm
import time
from torch.optim import SGD, Adam, AdamW
import torch.nn.functional as F
import matplotlib.colors as mcolors
import matplotlib.cm as cm

In [ ]:
folder = '\\\\NAS_LOCCO\\Amaury\\DATA\\polMFM_experimental_processed\\these_4polar_MFM\\2026_02_02_X'
#folder = '\\\\NAS_LOCCO\\Amaury\\DATA\\polMFM_experimental_processed\\these_4polar_MFM\\2026_02_actin_moein_21'
frame, x, y, z, N_photons, rho, eta, delta, score, x_start, y_start, z_start, rho_start, delta_start, batch = [], [], [], [], [], [], [], [], [], [], [], [], [], [], []
background_array_found = np.zeros((0,6))
ii=0
for filename in sorted(os.listdir(folder), key=lambda x: int(x.replace('.npz',''))):
    data = np.load(rf"{folder}\{filename}")
    batch.append(ii + np.arange(len(data['frame'])))
    ii+=len(data['frame'])
    frame = np.concatenate((frame, data['frame']))
    x = np.concatenate((x, data['x']))
    y = np.concatenate((y, data['y']))
    z = np.concatenate((z, data['z'])) 
    background_array_found = np.concatenate((background_array_found, data['background_array_found'].reshape(len(data['frame']), 6))) 
    N_photons = np.concatenate((N_photons, data['N_photons']))
    rho = np.concatenate((rho, data['rho']))
    eta = np.concatenate((eta, data['eta']))
    delta = np.concatenate((delta, data['delta']))
    score = np.concatenate((score, data['score']))
    x_start = np.concatenate((x_start, data['x_start']))
    y_start = np.concatenate((y_start, data['y_start']))
    z_start = np.concatenate((z_start, data['z_start']))    
    rho_start = np.concatenate((rho_start, data['rho_start']))
    delta_start = np.concatenate((delta_start, data['delta_start']))
frame = np.array(frame).astype(int)
x0 = np.array(x)
y0 = np.array(y)
z0 = np.array(z)
N_photons = np.array(N_photons)
rho0 = np.array(rho)
eta0 = np.array(eta)
delta0 = np.array(delta)
score = np.array(score)
rho_start = np.array(rho_start)
delta_start = np.array(delta_start)
z_start = np.array(z_start)
x_start = np.array(x_start)
y_start = np.array(y_start)
background_array_found = np.array(background_array_found)

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using GPU")
else:   
    device = torch.device('cpu')
    print("Using CPU")

In [ ]:
d = np.array([1.2,1.2,1.2])
d = np.array([np.mean(d)-0.385, np.mean(d), np.mean(d)+0.385])
QE = 0.92
EM = 200
sensitivity = 15.4

In [ ]:
Nframe=400
raw = np.zeros((Nframe,6,214,129))

In [ ]:
path_info = '\\\\NAS_LOCCO\\Amaury\\DATA\\4polar_data_raw\\2026_02_02_SLB_1um\\SM_tres_haut\\Calib_Polar_2026-02-02\\images\\RAW_DATA\\image_Pos0.ome_results_fr1to15000_method=Propagation matrix_box-method=Fixed_box5.csv'
#path_info = '\\\\NAS_LOCCO\\Amaury\\DATA\\4polar_data_raw\\2026_03_04_actin_moein_lola\\sample5\\sm\\Calib_Polar_2026-03-04\\images\\RAW_DATA\\positions.csv'

In [ ]:
def extract_frames(frame_0, N_frame):
    error_indices = []
    print('extracting frame '+str(frame_0)+' to '+str(frame_0+N_frame-1))
    for i in range(N_frame):
        number = str(frame_0 + i).zfill(5)
       #path_data = '\\\\NAS_LOCCO\\Amaury\\DATA\\4polar_data_raw\\2026_03_04_actin_moein_lola\\sample5\\sm\\Calib_Polar_2026-03-04\\images\\RAW_DATA\\image_Pos0_reco\\image_Pos0_'+number+'.tif'
        path_data = '\\\\NAS_LOCCO\\Amaury\\DATA\\4polar_data_raw\\2026_02_02_SLB_1um\\SM_tres_haut\\Calib_Polar_2026-02-02\\images\\RAW_DATA\\image_Pos0_reco_concat\\image_Pos0_'+number+'.tif'
        raw_ = extract_raw(path_data)
        if raw_ is None:
            error_indices.append(i)
            continue
        else:
            raw[i] = raw_
        del(raw_)
    return raw, error_indices

def extract_positions(frame_0, N_frame, error_indices):
    index_frame = []
    x, y, z, rho, delta = [], [], [], [], []
    ind = 0
    for i in range(N_frame):
        if i not in error_indices:
            x__, y__, z__, rho__, delta__ = position_from_data(data, frame_0+i)
            x = np.concatenate((x, x__))
            y = np.concatenate((y, y__))
            z = np.concatenate((z, z__))
            rho = np.concatenate((rho, rho__))
            delta = np.concatenate((delta, delta__))
            for k in range(len(x__)):
                index_frame.append(ind)
        ind+=1
    index_frame=np.array(index_frame)
    return x, y, z, rho, delta, index_frame

def limit(x, lim, slope, upper=True):
    '''
    if upper:
       return torch.sum(torch.tensor(1/(1+torch.exp(-slope*(x-lim))), requires_grad=True, device=device))
    else:
        return torch.sum(torch.tensor(1/(1+torch.exp(slope*(x-lim))), requires_grad=True, device=device))
    '''
    if upper:
        return torch.sum(torch.exp((x-lim)*slope))
    else:
        return torch.sum(torch.exp(-1*(x-lim)*slope))
    
def loss_aberrations(zernx, zerny, x_, y_, z_, N_, rho_, eta_, delta_, bck_, data_, second_plane, sigma, dim_simu):
    dim_data = 6
    M_ = compute_M(xp=x_, yp=y_, zp=z_, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_masky=phase_mask, phase_maskx=phase_mask
                   , zernike_base=zernike_base, zernike_coefs_x=torch.reshape(zernx, (3,15)), zernike_coefs_y=torch.reshape(zerny, (3,15)),
                        second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, device=device, mode=mode)
    h = PSF(rho=rho_, eta=eta_, delta=delta_, M=M_, N_photons=N_)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    loss = torch.sum(torch.add(h, -(data_+sigma**2)*torch.log(h+torch.reshape(bck_, (h.shape[0],3,2))[:, :, :, None, None]+sigma**2)))
    return loss
    
def score_eval(M_, rho, eta, delta, N_photons, data, background, sigma, dim_simu):
    dim_data = 6
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    score = torch.sum(torch.add(h, -(data+sigma**2)*torch.log(h+background+sigma**2)), dim=(1,2,3,4))
    return score.numpy() 

In [ ]:
zernx = torch.tensor(np.arange(3*15))
torch.reshape(zernx, (3,15))

In [ ]:
data = pos_from_csv(path_info)

In [ ]:
N_batch = 5000
batch_offset = 0

In [ ]:
mode = 'Stokes'
''' J_dichroic = np.array([[0.7838338      ,               -0.25981125 + 1j*  -0.48329058],[
      -0.4230177 + 1j*  0.27765664  ,   -0.7788276 + 1j*  -0.28660256
]]) # this one is for the abstract

J1 = np.array([[ 1.2604369        ,             -0.44922367 + 1j*  0.6776327 ],[
      -0.40610462 + 1j*  0.6554575  ,   -1.2775537 + 1j*  0.034650166 ]])
J2 = np.array([[ 0.57820666               ,      -1.2006966 + 1j*  0.67720294 ],[
      1.0710695 + 1j*  0.8422108   ,  0.2654659 + 1j*  0.58898413 ]])
'''
J1 = np.array([[ 0.86491126        ,             -0.39550093 + 1j*  -0.5777423 ],[
      -0.5027361 + 1j*  -0.44507027  ,   -0.857308 + 1j*  0.2078727 ]])
J2 = np.array([[ 0.34593397               ,      -0.58738905 + 1j*  -0.8697098 ],[
      0.6510633 + 1j*  0.82235897   ,  -0.09731049 + 1j*  0.3847822 ]])
J_dichroic = np.array([J1, J2, J1])

In [ ]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
# Filtrer

In [ ]:
res = []
for batch_number in range(N_batch):
    t0 = time.time()
    # extracteing the raw 6-stack tiff files
    raw, error_indices = extract_frames((batch_number+batch_offset)*Nframe+1, Nframe)
    # extracting the position from Louise pipeline
    x, y, z, rho, delta, index_frame = extract_positions((batch_number+batch_offset)*Nframe+1, Nframe, error_indices)
    # converting to photon count
    raw = raw*sensitivity/(QE*EM)
    # these quantites are used to evaluated the noise and inserted into the loss
    sigma = np.std(raw.flatten())
    background = np.mean(raw.flatten())
    print(background, sigma)
    #print(index_frame)
    # removing all the PSF where a parameter is evaluated to nan in Louise pipeline
    nb = len(x)

    for k, ele in enumerate(x):
        if np.isnan(x[nb-1-k]) or np.isnan(y[nb-1-k]) or np.isnan(z[nb-1-k]) or np.isnan(rho[nb-1-k]) or np.isnan(delta[nb-1-k]):
            x = np.delete(x,nb-1-k,0)
            y = np.delete(y,nb-1-k,0)
            z = np.delete(z,nb-1-k,0)
            rho = np.delete(rho,nb-1-k,0)
            delta = np.delete(delta,nb-1-k,0)
            index_frame = np.delete(index_frame,nb-1-k,0)
    # extracting the psf from the files
    single_psf = extract_raw_xy(raw[0], x[index_frame==0], y[index_frame==0])
    for i in range(1,Nframe):
        single_psf = np.concatenate((single_psf, extract_raw_xy(raw[i], x[index_frame==i], y[index_frame==i])))

    # dimenstion matching to have x in horizontal and y in vertical when considering what appears in a tiff file
    single_psf = single_psf[:,::-1]
    #single_psf = np.transpose(single_psf[:,:,:,:,:], axes=(0,1,2,4,3))
    single_psf = single_psf[:,:,:,::-1,:]
    #x = np.max(x)-x
    x, y = y, -x

    # removing all the PSF where there are two emitters, either too bright or the middle plane less bright than the extremal ones
    nb = len(x)
    
    NPSF = len(x)
    print(NPSF)
    mask = (frame>=(batch_number+batch_offset)*Nframe) & (frame<=(batch_number+batch_offset+1)*Nframe-1)
    # retireved params
    x_ = torch.tensor((x0[mask]-(x/120).astype(int)*120)/1000, device=device)
    y_ = torch.tensor((y0[mask]-(y/120).astype(int)*120)/1000,  device=device)
    z_ = torch.tensor(z0[mask],  device=device)/1000
    rho_ = torch.tensor(rho0[mask], device=device)
    eta_ = torch.tensor(eta0[mask],  device=device)
    delta_ =  torch.tensor(delta0[mask],  device=device) 
    N_ =  torch.tensor(N_photons[mask],  device=device) 
    bck_ =  torch.tensor(background_array_found[mask],  device=device)
    
    filtering = (z_>0.4) & (z_<1.2) & (N_<7000)
    x_ = x_[filtering]
    y_ = y_[filtering]
    z_ = z_[filtering]
    rho_ = rho_[filtering]
    eta_ = eta_[filtering]
    delta_ = delta_[filtering]
    N_ = N_[filtering]
    bck_ = bck_[filtering]
    
    # microscope parameters
    d_ = -torch.tensor([d[1] for k in range(len(x_))], requires_grad=False, device=device)
    second_plane = torch.tensor([d[1]-d[0], 0, d[1]-d[2]], device=device, requires_grad=False)
    polar_projections = np.array([0, 45, 0])

    if batch_number==0:
        N=torch.tensor(80, device=device, requires_grad=False)
        l_pixel=torch.tensor(16, device=device, requires_grad=False)
        NA=torch.tensor(1.4, device=device, requires_grad=False)
        mag=torch.tensor(100, device=device, requires_grad=False)
        lambd=torch.tensor(638, device=device, requires_grad=False)
        f_tube=torch.tensor(200, device=device, requires_grad=False)
        MAG=torch.tensor(200/150, device=device, requires_grad=False)
        
        SAF = False
    
        if SAF:
            xx, yy, th1, phi, [Ex0, Ex1, Ex2], [Ey0, Ey1, Ey2], r, r_cut, r_cut_saf, k_, f_o, costh2 = vectorial_BFP_perfect_focus(N, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, SAF=SAF, device=device, J_dichroic=J_dichroic)
        else:
            costh2=None
            xx, yy, th1, phi, [Ex0, Ex1, Ex2], [Ey0, Ey1, Ey2], r, r_cut, k_, f_o = vectorial_BFP_perfect_focus(N, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, SAF=SAF, device=device, J_dichroic=J_dichroic)
    
        u, v, Npadding = padding(r, r_cut, k_, f_o,  N=N, l_pixel=l_pixel, NA=NA, mag=mag, lambd=lambd, 
                  f_tube=f_tube, MAG=MAG, device=device)

        phase_mask = torch.stack([torch.ones((N,N), device=device), torch.ones((N,N), device=device), torch.ones((N,N), device=device)])
        zernike_base = generate_zernike_base(r_cut=r_cut, N=N, zernike_order=4, device=device)
        zernike_coefs_x = torch.zeros((3,15), device=device).to(torch.complex64)*0.00000001
        zernike_coefs_y = torch.zeros((3,15), device=device).to(torch.complex64)
        
        xx = pad(xx, Npadding).to(torch.complex64).detach()
        yy = pad(yy, Npadding).to(torch.complex64).detach()
        th1 = pad(th1, Npadding).to(torch.complex64).detach()
        phi = pad(phi, Npadding).to(torch.complex64).detach()
        Ex0 = pad(Ex0, Npadding).to(torch.complex64).detach()
        Ex1 = pad(Ex1, Npadding).to(torch.complex64).detach()
        Ex2 = pad(Ex2, Npadding).to(torch.complex64).detach()
        Ey0 = pad(Ey0, Npadding).to(torch.complex64).detach()
        Ey1 = pad(Ey1, Npadding).to(torch.complex64).detach()
        Ey2 = pad(Ey2, Npadding).to(torch.complex64).detach()
        phase_mask = pad(phase_mask, Npadding).to(torch.complex64).detach()
        zernike_base = pad(zernike_base, Npadding).to(torch.complex64).detach()
        if SAF:
            costh2 = pad(costh2, Npadding).to(torch.complex64).detach()

    # convert to tensor
    noisy_psf = torch.tensor(np.array([single_psf[k] for k in range(len(x))]), device=device, dtype=torch.float32)
    noisy_psf = noisy_psf[filtering]
    Mtest = compute_M(xp=x_, yp=y_, zp=z_, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=zernike_base, zernike_coefs_x=zernike_coefs_x, zernike_coefs_y=zernike_coefs_y,
                        second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, device=device, mode=mode)
    
    h = PSF(rho=rho_, eta=eta_, delta=delta_, M=Mtest, N_photons=N_).to(torch.float32)
    dim_simu = int(h.shape[-1]//2)
    dim_data=6
    h = h[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    if False:
        for nb in range(noisy_psf.shape[0]):
            maxi = max(np.max(noisy_psf[nb,:,:].flatten().cpu().detach().numpy()), np.max(h[nb,:,:].flatten().cpu().detach().numpy()))
            fig, ax = plt.subplots(3,2)
            ax[0,0].imshow(noisy_psf[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,0].imshow(noisy_psf[nb,1,0].cpu().detach().numpy() , vmin=0., vmax=maxi, cmap='gray')
            ax[1,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,0].imshow(noisy_psf[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[0,1].imshow(noisy_psf[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,1].imshow(noisy_psf[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,1].imshow(noisy_psf[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            plt.show()
            del(fig, ax)
            fig, ax = plt.subplots(3,2)
            ax[0,0].imshow(h[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,0].imshow(h[nb,1,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,0].imshow(h[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[0,1].imshow(h[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,1].imshow(h[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,1].imshow(h[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            plt.show()
    
    params = torch.tensor([0 for u in range(2*3*15)]).to(torch.float32).to(device)
    params.requires_grad=True
    optimizer = torch.optim.Adam([params], lr=0.1)
    num_epochs_max = 80
    loss_ = []
    values = []
    for i in tqdm(range(num_epochs_max)):
        optimizer.zero_grad()  # Reset gradients
        loss = loss_aberrations(params[:3*15], params[3*15:], x_, y_, z_, N_, rho_, eta_, delta_, bck_,
                           noisy_psf, second_plane, sigma, dim_simu)
        loss_.append(loss.cpu().detach().numpy())
        values.append(params.cpu().detach().numpy())
        loss.backward()
        optimizer.step()
    fig, ax = plt.subplots(2)
    ax[0].plot(loss_)
    ax[1].plot(values)
    plt.show()
    del(ax, loss_, values)
    res.append(params.cpu().detach().numpy().reshape(6,15))
    Mtest = compute_M(xp=x_, yp=y_, zp=z_, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=zernike_base
                      , zernike_coefs_x=params[:3*15].reshape((3,15)), zernike_coefs_y=params[3*15:].reshape((3,15)),
                        second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, device=device, mode=mode)
    
    h = PSF(rho=rho_, eta=eta_, delta=delta_, M=Mtest, N_photons=N_).to(torch.float32)
    dim_simu = int(h.shape[-1]//2)
    dim_data=6
    h = h[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    if False:
        for nb in range(noisy_psf.shape[0]):
            maxi = max(np.max(noisy_psf[nb,:,:].flatten().cpu().detach().numpy()), np.max(h[nb,:,:].flatten().cpu().detach().numpy()))
            fig, ax = plt.subplots(3,2)
            ax[0,0].imshow(noisy_psf[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,0].imshow(noisy_psf[nb,1,0].cpu().detach().numpy() , vmin=0., vmax=maxi, cmap='gray')
            ax[1,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,0].imshow(noisy_psf[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[0,1].imshow(noisy_psf[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,1].imshow(noisy_psf[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,1].imshow(noisy_psf[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            plt.show()
            del(fig, ax)
            fig, ax = plt.subplots(3,2)
            ax[0,0].imshow(h[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,0].imshow(h[nb,1,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,0].imshow(h[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,0].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[0,1].imshow(h[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,1].imshow(h[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,1].imshow(h[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,1].scatter(x_[nb].cpu().detach().numpy()/0.120+6, y_[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            plt.show()

In [ ]:
res = np.array(res)

In [ ]:
res.shape

In [ ]:
plt.rcParams['figure.figsize'] = [10,7]
mini = np.min(res)
maxi = np.max(res)
fig, ax = plt.subplots(2,3)
ax[0,0].violinplot(res[:,0])
ax[0,0].plot(np.arange(1,16), np.median(res, axis=0)[0])
ax[0,0].set_ylim((mini, maxi))
ax[0,0].grid()
ax[0,1].violinplot(res[:,1])
ax[0,1].plot(np.arange(1,16), np.median(res, axis=0)[1])
ax[0,1].set_ylim((mini, maxi))
ax[0,1].grid()
ax[0,2].violinplot(res[:,2])
ax[0,2].plot(np.arange(1,16), np.median(res, axis=0)[2])
ax[0,2].set_ylim((mini, maxi))
ax[0,2].grid()
ax[1,0].violinplot(res[:,3])
ax[1,0].plot(np.arange(1,16), np.median(res, axis=0)[3])
ax[1,0].set_ylim((mini, maxi))
ax[1,0].grid()
ax[1,1].violinplot(res[:,4])
ax[1,1].plot(np.arange(1,16), np.median(res, axis=0)[4])
ax[1,1].set_ylim((mini, maxi))
ax[1,1].grid()
ax[1,2].violinplot(res[:,5])
ax[1,2].plot(np.arange(1,16), np.median(res, axis=0)[5])
ax[1,2].set_ylim((mini, maxi))
ax[1,2].grid()
plt.show()

In [ ]:
np.mean(res, axis=0)

In [ ]:
np.where(N_.cpu().detach().numpy()>10000)

In [ ]:
psf = np.sum(noisy_psf[100].cpu().detach().numpy(), axis=(0,1))
im = plt.imshow(psf)
cb = plt.colorbar(im)

In [ ]:
bck = (np.mean(psf[:2,:])+np.mean(psf[-2:,:])+np.mean(psf[:,:2])+np.mean(psf[:,-2:]))/4

In [ ]:
bck

In [ ]:
total = np.sum(psf)

In [ ]:
n = total - 13*13*bck

In [ ]:
n

In [ ]:
N_[40]

In [ ]:
nb = []
for i in range(740):
    psf = np.sum(noisy_psf[i].cpu().detach().numpy(), axis=(0,1))
    bck = (np.mean(psf[:2,:])+np.mean(psf[-2:,:])+np.mean(psf[:,:2])+np.mean(psf[:,-2:]))/4
    total = np.sum(psf)
    n = total - 13*13*bck
    nb.append(n)

In [ ]:
hh = plt.hist(nb, bins=50)
plt.grid()

In [ ]:
hh = plt.hist(N_.cpu(), bins=50)

In [ ]:
hh = plt.hist(N_.cpu().detach().numpy()[:740]-np.array(nb), bins=50)

In [ ]:
np.mean(raw[0,0].flatten())

In [ ]:
im = plt.imshow(raw[40,5])
cb = plt.colorbar(im)